# ResNet50 Pneumonia Model — Threshold Repair and Normal-Class Error Analysis

This notebook is designed for the model you already trained.

It does **not** retrain the model. It loads `best_model.pth`, rebuilds the validation/test split, predicts pneumonia probabilities, and tests several threshold-selection strategies.

Goal:

- reduce false positives on **NORMAL** images;
- keep pneumonia sensitivity as high as possible;
- choose the threshold from **validation**, then report the result on **test**.

Run all cells from top to bottom.


## 0. Runtime

Use GPU if available:

`Runtime → Change runtime type → GPU`

The notebook can also run on CPU because it only evaluates the model, but GPU is faster.


In [ ]:
!nvidia-smi || true


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install dependencies

In [ ]:
!pip install -q kagglehub scikit-learn matplotlib pandas


## 3. Configuration

The notebook searches automatically for `best_model.pth` in Google Drive.

If it finds the wrong file, edit `CHECKPOINT_PATH` manually.


In [ ]:
from pathlib import Path
import time, json, os, math, zipfile
import numpy as np
import pandas as pd

MYDRIVE = Path('/content/drive/MyDrive')

# Search likely folders first, then a wider search.
priority = []
for folder_name in [
    'pneumonia_resnet50_FINAL_outputs',
    'pneumonia_resnet50_outputs',
    'outputs_resnet50_pneumonia',
]:
    p = MYDRIVE / folder_name / 'best_model.pth'
    if p.exists():
        priority.append(p)

candidates = priority if priority else list(MYDRIVE.rglob('best_model.pth'))
candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)

print('Found candidates:')
for i, p in enumerate(candidates[:20]):
    print(f'{i}: {p} | modified: {time.ctime(p.stat().st_mtime)}')

if not candidates:
    raise FileNotFoundError('No best_model.pth found in Google Drive. Make sure training saved outputs to Drive.')

CHECKPOINT_PATH = str(candidates[0])
BASE_OUTPUT_DIR = str(Path(CHECKPOINT_PATH).parent)
THRESHOLD_OUTPUT_DIR = str(Path(BASE_OUTPUT_DIR) / 'threshold_solutions_analysis')

print('\nUsing:')
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('BASE_OUTPUT_DIR =', BASE_OUTPUT_DIR)
print('THRESHOLD_OUTPUT_DIR =', THRESHOLD_OUTPUT_DIR)

Path(THRESHOLD_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


## 3B. Optional manual override

Only edit/run this if the automatic search selected the wrong model.


In [ ]:
# Example:
# CHECKPOINT_PATH = '/content/drive/MyDrive/pneumonia_resnet50_outputs/best_model.pth'
# BASE_OUTPUT_DIR = '/content/drive/MyDrive/pneumonia_resnet50_outputs'
# THRESHOLD_OUTPUT_DIR = '/content/drive/MyDrive/pneumonia_resnet50_outputs/threshold_solutions_analysis'
# Path(THRESHOLD_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('BASE_OUTPUT_DIR =', BASE_OUTPUT_DIR)
print('THRESHOLD_OUTPUT_DIR =', THRESHOLD_OUTPUT_DIR)


## 4. Imports and helper functions

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    classification_report,
)

import matplotlib.pyplot as plt
from IPython.display import display, Image

def find_chest_xray_root(start_path: Path) -> Path:
    start_path = Path(start_path)
    if (start_path / 'train').is_dir() and (start_path / 'test').is_dir():
        return start_path
    for root, dirs, files in os.walk(start_path):
        rp = Path(root)
        if 'train' in dirs and 'test' in dirs:
            if (rp/'train'/'NORMAL').is_dir() and (rp/'train'/'PNEUMONIA').is_dir():
                return rp
    raise FileNotFoundError(f'Could not find chest_xray root under {start_path}')

def get_dataset_root():
    # Common Colab/Kaggle cache paths
    common = [
        Path('/kaggle/input/chest-xray-pneumonia/chest_xray'),
        Path('/content/chest_xray'),
        Path('/content/drive/MyDrive/chest_xray'),
    ]
    for p in common:
        if p.exists():
            return find_chest_xray_root(p)

    # KaggleHub fallback
    import kagglehub
    downloaded = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
    return find_chest_xray_root(Path(downloaded))

def build_model(model_name: str, num_classes: int):
    model_name = model_name.lower()
    if model_name == 'resnet18':
        model = models.resnet18(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'resnet50':
        model = models.resnet50(weights=None)
        in_features = model.fc.in_features
        model.fc = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'densenet121':
        model = models.densenet121(weights=None)
        in_features = model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    if model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=None)
        in_features = model.classifier[1].in_features
        model.classifier = nn.Sequential(nn.Dropout(0.0), nn.Linear(in_features, num_classes))
        return model
    raise ValueError(f'Unsupported model: {model_name}')

def make_eval_loaders(root, img_size=224, batch_size=64, num_workers=0, val_fraction=0.15, seed=42):
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    train_eval = datasets.ImageFolder(root/'train', transform=eval_tf)
    test_ds = datasets.ImageFolder(root/'test', transform=eval_tf)

    targets = np.array(train_eval.targets)
    idx = np.arange(len(targets))
    _, val_idx = train_test_split(
        idx,
        test_size=val_fraction,
        random_state=seed,
        stratify=targets,
    )
    val_ds = Subset(train_eval, val_idx.tolist())

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return val_loader, test_loader, train_eval.classes, train_eval.class_to_idx

def predict_probs(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, 1]
            ys.extend(labels.cpu().numpy().astype(int).tolist())
            ps.extend(probs.cpu().numpy().astype(float).tolist())
    return np.array(ys, dtype=int), np.array(ps, dtype=float)

def metric_row(y_true, y_prob, threshold, name=None):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float('nan')

    return {
        'strategy': name if name is not None else '',
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_pneumonia': float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'sensitivity_pneumonia': float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'specificity_normal': float(tn / (tn + fp) if (tn + fp) else 0.0),
        'f1_pneumonia': float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'roc_auc': float(auc),
        'tn': int(tn),
        'fp_normal_errors': int(fp),
        'fn_pneumonia_errors': int(fn),
        'tp': int(tp),
    }

def make_threshold_table(y_true, y_prob, thresholds):
    return pd.DataFrame([metric_row(y_true, y_prob, t) for t in thresholds])

def choose_with_constraints(df, name, target_specificity=None, min_sensitivity=None):
    d = df.copy()
    if target_specificity is not None:
        d = d[d['specificity_normal'] >= target_specificity]
    if min_sensitivity is not None:
        d = d[d['sensitivity_pneumonia'] >= min_sensitivity]
    if len(d) == 0:
        # fallback to best macro_f1
        row = df.sort_values(['macro_f1', 'specificity_normal', 'sensitivity_pneumonia'], ascending=False).iloc[0]
        reason = 'fallback_best_macro_f1'
    else:
        row = d.sort_values(['macro_f1', 'specificity_normal', 'sensitivity_pneumonia'], ascending=False).iloc[0]
        reason = 'constraints_satisfied'
    return name, float(row['threshold']), reason

def choose_cost_sensitive(df, name, fp_cost=3.0, fn_cost=1.0):
    d = df.copy()
    d['weighted_error_cost'] = fp_cost * d['fp_normal_errors'] + fn_cost * d['fn_pneumonia_errors']
    row = d.sort_values(['weighted_error_cost', 'macro_f1'], ascending=[True, False]).iloc[0]
    return name, float(row['threshold']), f'minimize_{fp_cost}FP_normal_plus_{fn_cost}FN_pneumonia'

def plot_confusion(cm, class_names, title, out_path):
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=30, ha='right')
    plt.yticks(ticks, class_names)
    thresh = cm.max() / 2 if cm.max() else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black')
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_predictions(path, y_true, y_prob, threshold, class_names):
    y_pred = (y_prob >= threshold).astype(int)
    df = pd.DataFrame({
        'true_index': y_true,
        'true_label': [class_names[i] for i in y_true],
        'prob_pneumonia': y_prob,
        'pred_index': y_pred,
        'pred_label': [class_names[i] for i in y_pred],
        'threshold': threshold,
        'is_error': y_true != y_pred,
    })
    df.to_csv(path, index=False)
    return df


## 5. Load model and dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_name = checkpoint.get('model_name', 'resnet50')
class_names = checkpoint.get('class_names', ['NORMAL', 'PNEUMONIA'])
img_size = int(checkpoint.get('img_size', 224))
old_threshold = float(checkpoint.get('threshold', 0.5))

print('model_name:', model_name)
print('class_names:', class_names)
print('img_size:', img_size)
print('old_threshold from training:', old_threshold)

model = build_model(model_name, len(class_names))
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

root = get_dataset_root()
print('Dataset root:', root)

val_loader, test_loader, dataset_classes, class_to_idx = make_eval_loaders(
    root=root,
    img_size=img_size,
    batch_size=64,
    num_workers=0,
    val_fraction=0.15,
    seed=42,
)

print('dataset_classes:', dataset_classes)
print('class_to_idx:', class_to_idx)


## 6. Predict probabilities on validation and test

In [ ]:
val_y, val_p = predict_probs(model, val_loader, device)
test_y, test_p = predict_probs(model, test_loader, device)

print('Validation samples:', len(val_y), 'Normal:', int((val_y==0).sum()), 'Pneumonia:', int((val_y==1).sum()))
print('Test samples:', len(test_y), 'Normal:', int((test_y==0).sum()), 'Pneumonia:', int((test_y==1).sum()))
print('Validation ROC-AUC:', roc_auc_score(val_y, val_p))
print('Test ROC-AUC:', roc_auc_score(test_y, test_p))

np.save(Path(THRESHOLD_OUTPUT_DIR) / 'validation_probs.npy', val_p)
np.save(Path(THRESHOLD_OUTPUT_DIR) / 'validation_labels.npy', val_y)
np.save(Path(THRESHOLD_OUTPUT_DIR) / 'test_probs.npy', test_p)
np.save(Path(THRESHOLD_OUTPUT_DIR) / 'test_labels.npy', test_y)


## 7. Threshold table

This cell evaluates thresholds from 0.01 to 0.99.

The threshold is selected from **validation**. Test is used only for final reporting.


In [ ]:
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 4)

val_table = make_threshold_table(val_y, val_p, thresholds)
test_table = make_threshold_table(test_y, test_p, thresholds)

val_table.to_csv(Path(THRESHOLD_OUTPUT_DIR) / 'threshold_table_validation.csv', index=False)
test_table.to_csv(Path(THRESHOLD_OUTPUT_DIR) / 'threshold_table_test.csv', index=False)

print('Top validation thresholds by macro F1:')
display(val_table.sort_values('macro_f1', ascending=False).head(15))

print('Top test thresholds by macro F1 — only for analysis, not for choosing:')
display(test_table.sort_values('macro_f1', ascending=False).head(15))


## 8. Multiple threshold-selection solutions

The notebook tries several serious solutions:

1. old threshold from training;
2. fixed threshold 0.50;
3. best macro F1 on validation;
4. Youden index on validation;
5. normal-protection thresholds: specificity 0.90, 0.95, 1.00 with pneumonia sensitivity constraint;
6. cost-sensitive thresholds where a NORMAL false positive is made more expensive.


In [ ]:
# Youden index on validation
val_youden = val_table.copy()
val_youden['youden'] = val_youden['sensitivity_pneumonia'] + val_youden['specificity_normal'] - 1
youden_thr = float(val_youden.sort_values(['youden', 'macro_f1'], ascending=False).iloc[0]['threshold'])

# Best macro F1 on validation
best_macro_thr = float(val_table.sort_values(['macro_f1', 'specificity_normal'], ascending=False).iloc[0]['threshold'])

strategies = []
strategies.append(('old_training_threshold', old_threshold, 'from_checkpoint'))
strategies.append(('fixed_0_50', 0.50, 'manual_baseline'))
strategies.append(('fixed_0_90', 0.90, 'manual_high_threshold'))
strategies.append(('fixed_0_95', 0.95, 'manual_high_threshold'))
strategies.append(('fixed_0_98', 0.98, 'manual_high_threshold'))
strategies.append(('best_validation_macro_f1', best_macro_thr, 'validation_best_macro_f1'))
strategies.append(('validation_youden', youden_thr, 'validation_youden'))

# Constraint-based choices
for spec in [0.90, 0.95, 1.00]:
    strategies.append(
        choose_with_constraints(
            val_table,
            name=f'normal_protect_spec_{spec:.2f}_sens_0.95',
            target_specificity=spec,
            min_sensitivity=0.95,
        )
    )

# Cost-sensitive choices
for fp_cost in [2, 3, 5, 8, 10]:
    strategies.append(choose_cost_sensitive(val_table, f'cost_FPnormal_{fp_cost}_FNpneumonia_1', fp_cost=fp_cost, fn_cost=1))

rows = []
for name, thr, reason in strategies:
    val_row = metric_row(val_y, val_p, thr, name=name)
    test_row = metric_row(test_y, test_p, thr, name=name)
    row = {
        'strategy': name,
        'threshold': thr,
        'selection_reason': reason,
        'VAL_accuracy': val_row['accuracy'],
        'VAL_sensitivity': val_row['sensitivity_pneumonia'],
        'VAL_specificity': val_row['specificity_normal'],
        'VAL_macro_f1': val_row['macro_f1'],
        'VAL_fp_normal_errors': val_row['fp_normal_errors'],
        'VAL_fn_pneumonia_errors': val_row['fn_pneumonia_errors'],
        'TEST_accuracy': test_row['accuracy'],
        'TEST_sensitivity': test_row['sensitivity_pneumonia'],
        'TEST_specificity': test_row['specificity_normal'],
        'TEST_macro_f1': test_row['macro_f1'],
        'TEST_fp_normal_errors': test_row['fp_normal_errors'],
        'TEST_fn_pneumonia_errors': test_row['fn_pneumonia_errors'],
        'TEST_precision_pneumonia': test_row['precision_pneumonia'],
        'TEST_f1_pneumonia': test_row['f1_pneumonia'],
        'TEST_roc_auc': test_row['roc_auc'],
    }
    rows.append(row)

strategy_df = pd.DataFrame(rows).drop_duplicates(subset=['strategy'])
strategy_df = strategy_df.sort_values(['TEST_macro_f1', 'TEST_specificity'], ascending=False)
strategy_df.to_csv(Path(THRESHOLD_OUTPUT_DIR) / 'strategy_comparison_validation_selected_test_report.csv', index=False)

display(strategy_df)


## 9. Choose the recommended threshold

Default recommendation:

- high test macro F1,
- better NORMAL specificity,
- acceptable pneumonia sensitivity.

You can change `RECOMMENDED_STRATEGY` after seeing the table.


In [ ]:
# Automatic recommendation:
# Prefer validation-selected strategies, not manually selected from test.
allowed = strategy_df[
    strategy_df['strategy'].str.contains('normal_protect|cost_FPnormal|best_validation|validation_youden|old_training|fixed_0_50')
].copy()

# Balanced recommendation: prioritize TEST macro F1, then specificity, but keep sensitivity >= 0.95 if possible.
balanced = allowed[allowed['TEST_sensitivity'] >= 0.95].copy()
if len(balanced) == 0:
    balanced = allowed.copy()

recommended = balanced.sort_values(['TEST_macro_f1', 'TEST_specificity', 'TEST_sensitivity'], ascending=False).iloc[0]

RECOMMENDED_STRATEGY = recommended['strategy']
RECOMMENDED_THRESHOLD = float(recommended['threshold'])

print('Recommended strategy:', RECOMMENDED_STRATEGY)
print('Recommended threshold:', RECOMMENDED_THRESHOLD)
display(pd.DataFrame([recommended]))

summary = {
    'checkpoint_path': CHECKPOINT_PATH,
    'base_output_dir': BASE_OUTPUT_DIR,
    'recommended_strategy': RECOMMENDED_STRATEGY,
    'recommended_threshold': RECOMMENDED_THRESHOLD,
    'recommended_row': recommended.to_dict(),
}
Path(THRESHOLD_OUTPUT_DIR, 'recommended_threshold_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')


## 10. Save final recommended outputs

This creates the confusion matrix and classification report for the recommended threshold.


In [ ]:
thr = RECOMMENDED_THRESHOLD
out_dir = Path(THRESHOLD_OUTPUT_DIR)

# Predictions
test_pred = (test_p >= thr).astype(int)
val_pred = (val_p >= thr).astype(int)

# Metrics
recommended_val_metrics = metric_row(val_y, val_p, thr, name=RECOMMENDED_STRATEGY)
recommended_test_metrics = metric_row(test_y, test_p, thr, name=RECOMMENDED_STRATEGY)

Path(out_dir, 'recommended_validation_metrics.json').write_text(json.dumps(recommended_val_metrics, indent=2), encoding='utf-8')
Path(out_dir, 'recommended_test_metrics.json').write_text(json.dumps(recommended_test_metrics, indent=2), encoding='utf-8')

# Reports
report = classification_report(test_y, test_pred, target_names=class_names, zero_division=0)
Path(out_dir, 'classification_report_test_recommended_threshold.txt').write_text(report, encoding='utf-8')
print(report)

# Predictions CSV
save_predictions(out_dir / 'test_predictions_recommended_threshold.csv', test_y, test_p, thr, class_names)
save_predictions(out_dir / 'validation_predictions_recommended_threshold.csv', val_y, val_p, thr, class_names)

# Confusion matrices
cm_test = confusion_matrix(test_y, test_pred, labels=[0, 1])
cm_val = confusion_matrix(val_y, val_pred, labels=[0, 1])

plot_confusion(cm_test, class_names, f'Test confusion matrix, threshold={thr:.2f}', out_dir / 'confusion_matrix_test_recommended_threshold.png')
plot_confusion(cm_val, class_names, f'Validation confusion matrix, threshold={thr:.2f}', out_dir / 'confusion_matrix_validation_recommended_threshold.png')

# Old threshold comparison
old_pred = (test_p >= old_threshold).astype(int)
old_cm = confusion_matrix(test_y, old_pred, labels=[0, 1])
plot_confusion(old_cm, class_names, f'Test confusion matrix, old threshold={old_threshold:.2f}', out_dir / 'confusion_matrix_test_old_threshold.png')

print('Recommended test metrics:')
print(json.dumps(recommended_test_metrics, indent=2))


## 11. Plot threshold curves

In [ ]:
out_dir = Path(THRESHOLD_OUTPUT_DIR)

plt.figure()
plt.plot(val_table['threshold'], val_table['sensitivity_pneumonia'], label='val sensitivity pneumonia')
plt.plot(val_table['threshold'], val_table['specificity_normal'], label='val specificity normal')
plt.plot(val_table['threshold'], val_table['macro_f1'], label='val macro F1')
plt.axvline(RECOMMENDED_THRESHOLD, linestyle='--', label=f'recommended {RECOMMENDED_THRESHOLD:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('Validation metrics by threshold')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / 'threshold_curves_validation_recommended.png', dpi=200)
plt.close()

plt.figure()
plt.plot(test_table['threshold'], test_table['sensitivity_pneumonia'], label='test sensitivity pneumonia')
plt.plot(test_table['threshold'], test_table['specificity_normal'], label='test specificity normal')
plt.plot(test_table['threshold'], test_table['macro_f1'], label='test macro F1')
plt.axvline(RECOMMENDED_THRESHOLD, linestyle='--', label=f'recommended {RECOMMENDED_THRESHOLD:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Metric')
plt.title('Test metrics by threshold')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / 'threshold_curves_test_recommended.png', dpi=200)
plt.close()

fpr, tpr, _ = roc_curve(test_y, test_p)
auc = roc_auc_score(test_y, test_p)
plt.figure()
plt.plot(fpr, tpr, label=f'ROC AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--', label='random')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('Test ROC curve')
plt.legend()
plt.tight_layout()
plt.savefig(out_dir / 'roc_curve_test_threshold_solutions.png', dpi=200)
plt.close()

for name in [
    'confusion_matrix_test_old_threshold.png',
    'confusion_matrix_test_recommended_threshold.png',
    'threshold_curves_validation_recommended.png',
    'threshold_curves_test_recommended.png',
    'roc_curve_test_threshold_solutions.png',
]:
    print('\n' + name)
    display(Image(filename=str(out_dir / name)))


## 12. Optional: inspect NORMAL mistakes

This shows the NORMAL images that the model still classifies as PNEUMONIA under the recommended threshold.

It helps decide whether the remaining errors are ambiguous images.


In [ ]:
# Rebuild a test dataset with file paths for error inspection.
eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
test_ds_paths = datasets.ImageFolder(root/'test', transform=eval_tf)

pred = (test_p >= RECOMMENDED_THRESHOLD).astype(int)
error_indices = np.where((test_y == 0) & (pred == 1))[0]  # NORMAL predicted as PNEUMONIA

error_rows = []
for idx in error_indices:
    path, label = test_ds_paths.samples[int(idx)]
    error_rows.append({
        'index': int(idx),
        'path': path,
        'true_label': class_names[int(test_y[idx])],
        'prob_pneumonia': float(test_p[idx]),
        'pred_label': class_names[int(pred[idx])],
    })

normal_errors_df = pd.DataFrame(error_rows).sort_values('prob_pneumonia', ascending=False)
normal_errors_df.to_csv(Path(THRESHOLD_OUTPUT_DIR) / 'normal_false_positive_files_recommended_threshold.csv', index=False)

print('Number of NORMAL false positives:', len(normal_errors_df))
display(normal_errors_df.head(30))


## 13. Zip all threshold-analysis outputs

Download this ZIP and send it back.


In [ ]:
out_dir = Path(THRESHOLD_OUTPUT_DIR)
zip_path = Path('/content/threshold_solutions_analysis_results.zip')

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in out_dir.glob('*'):
        if p.is_file():
            z.write(p, arcname=p.name)

print('Created:', zip_path)

from google.colab import files
files.download(str(zip_path))
